In [1]:
import asyncio
import re
from typing import List, Dict
from google.oauth2 import service_account
from googleapiclient.discovery import build
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [ ]:

RAW_SHEET_NAME = "Sheet1" 
PROCESSED_SHEET_NAME = "Sheet2"  
CONTENT_DELIMITER = " | "
CATEGORIES = ["ABOUT_US", "EBOOK", "COURSES", "RECENT_BLOG", "TESTIMONIALS", "WEBINAR", "SERVICES", "PODCAST", "SHOP"]
COLUMN_TO_WRITE_URL_TO = {
    "ABOUT_US": "M", "EBOOK": "N", "COURSES": "O", "RECENT_BLOG": "P",
    "TESTIMONIALS": "Q", "WEBINAR": "R", "SERVICES": "S", "PODCAST": "T", "SHOP": "U"
}
EXTRACTION_METADATA_COLUMN = "V"
CATEGORY_THRESHOLDS = {
    "ABOUT_US": 200, "EBOOK": 200, "COURSES": 300, "RECENT_BLOG": 450, 
    "TESTIMONIALS": 100, "WEBINAR": 150, "SERVICES": 150, "PODCAST": 200, "SHOP": 100
}
OPENAI_API_KEY = ""
SHEET_URL = "https://docs.google.com/spreadsheets/d/1lmVQ8jnKWYowsfEG89FiiLIEAIkBs7uhshuOvMQ1tZc/edit?gid=0#gid=0"
CREDENTIALS_FILE = "data/url-to-email-445616-cebe4868914f.json"  

PROMPT_TEMPLATES = {
    "RECENT_BLOG": """
Compare the following blog posts based on the provided information. Evaluate each post on the following criteria, scoring each out of 10:

1. *Recency*: Score based on how recently the post was published, using the publication date provided.
2. *Relevance to Prospect's Industry*: Score based on how relevant the content is to the specified industry or keywords, using the excerpt provided.
3. *Post Length*: Score based on the word count, with a minimum of 200 words.

Use the scoring guidelines below for each criterion. After scoring, calculate a total score out of 30 for each post and rank them from best to worst.

{content}

**Scoring Guidelines:**
• *Recency*:
  - Within the last month: 10
  - 1-3 months ago: 8
  - 3-6 months ago: 6
  - 6-12 months ago: 4
  - Over 12 months ago: 2
• *Relevance to Prospect's Industry*:
  - Highly relevant (multiple keywords or strong topic alignment): 10
  - Moderately relevant (some keywords or partial alignment): 7
  - Slightly relevant (few keywords or weak alignment): 5
  - Not relevant: 0
• *Post Length*:
  - Less than 200 words: 0
  - 200-500 words: 5
  - 500-1000 words: 7
  - 1000-2000 words: 9
  - 2000+ words: 10

**Final Output:**
• Output ONLY the content text of the blog post with the highest score. Do NOT include scores, rankings, metadata (e.g., "Content length", "Line count", "Word Count"), or any other information. Do NOT explain the scoring or mention other posts.
"""
}

In [5]:
class GoogleSheetsManager:
    def __init__(self, credentials_file: str):
        scopes = ['https://www.googleapis.com/auth/spreadsheets']
        creds = service_account.Credentials.from_service_account_file(credentials_file, scopes=scopes)
        self.service = build('sheets', 'v4', credentials=creds)

    def extract_spreadsheet_id(self, sheet_url: str) -> str:
        pattern = r'/spreadsheets/d/([a-zA-Z0-9-_]+)'
        match = re.search(pattern, sheet_url)
        if match:
            return match.group(1)
        raise ValueError(f"Invalid Google Sheet URL: {sheet_url}")

# Parse content pieces
def parse_content_pieces(content: str, expected_count: int, category: str, row_num: int) -> List[str]:
    if not content or not content.strip():
        print(f"Warning: Empty content in row {row_num}, category {category}")
        return []
    cleaned_content = content.strip()
    print(f"Debug - Row {row_num}, {category}: Content preview: '{cleaned_content[:100]}...'")
    delimiters_to_try = [
        "-----NEXT CONTENT FROM HERE-----",
        "--- --NEXT CONTENT FROM HERE-----",
        "-----NEXT CONTENT FROM HERE--- --",
        " | ",
    ]
    pieces = None
    delimiter_used = None
    for delimiter in delimiters_to_try:
        if delimiter in cleaned_content:
            pieces = cleaned_content.split(delimiter)
            delimiter_used = delimiter
            break
    if pieces is None:
        pieces = [cleaned_content]
        print(f"Debug - Row {row_num}, {category}: No delimiter found, treating as single piece")
    else:
        print(f"Debug - Row {row_num}, {category}: Found {len(pieces)} pieces using delimiter '{delimiter_used}'")
    cleaned_pieces = [piece.strip() for piece in pieces if piece.strip()]
    if expected_count == 1 and len(cleaned_pieces) == 1:
        return cleaned_pieces
    elif expected_count > 1:
        if len(cleaned_pieces) >= expected_count:
            return cleaned_pieces[:expected_count]
        else:
            print(f"Warning: Expected {expected_count} pieces but found {len(cleaned_pieces)} in row {row_num}, category {category}")
            return cleaned_pieces
    else:
        print(f"Warning: Unexpected expected_count {expected_count} in row {row_num}, category {category}")
        return cleaned_pieces

async def read_data_from_sheet(spreadsheet_id: str, sheet_mgr: GoogleSheetsManager, categories: List[str]) -> List[Dict[str, List[str]]]:
    category_columns = {cat: COLUMN_TO_WRITE_URL_TO.get(cat.upper()) for cat in categories}
    if any(col is None for col in category_columns.values()):
        raise ValueError("One or more categories do not have a defined column in COLUMN_TO_WRITE_URL_TO")
    metadata_column = EXTRACTION_METADATA_COLUMN
    ranges = [f"{RAW_SHEET_NAME}!{col}2:{col}" for col in category_columns.values()] + [f"{RAW_SHEET_NAME}!{metadata_column}2:{metadata_column}"]
    try:
        response = await asyncio.to_thread(
            sheet_mgr.service.spreadsheets().values().batchGet(spreadsheetId=spreadsheet_id, ranges=ranges).execute
        )
    except Exception as e:
        raise RuntimeError(f"Failed to read data from spreadsheet {spreadsheet_id}: {str(e)}")
    value_ranges = response.get('valueRanges', [])
    column_data = {}
    for i, col in enumerate(category_columns.values()):
        column_data[col] = value_ranges[i].get('values', [])
    column_data[metadata_column] = value_ranges[-1].get('values', [])
    num_rows = max(len(values) for values in column_data.values()) if column_data else 0
    data = []
    for i in range(num_rows):
        row_data = {}
        metadata_value = column_data[metadata_column][i][0] if i < len(column_data[metadata_column]) and column_data[metadata_column][i] else ''
        print(f"Row {i+2} metadata: {metadata_value}")
        category_n_dict = {}
        for part in metadata_value.split(','):
            if '=' in part:
                cat, n_str = part.split('=', 1)
                try:
                    n = int(n_str.strip())
                    category_n_dict[cat.strip().upper()] = n
                except ValueError:
                    print(f"Warning: Invalid metadata '{part}' in row {i+2}")
        for category in categories:
            category_upper = category.upper()
            if category_upper in category_n_dict and category_n_dict[category_upper] > 0:
                col = category_columns[category]
                content = column_data[col][i][0] if i < len(column_data[col]) and column_data[col][i] else ''
                expected_count = category_n_dict[category_upper]
                parsed_pieces = parse_content_pieces(content, expected_count, category, i+2)
                if parsed_pieces:
                    row_data[category] = parsed_pieces
        if not row_data:
            print(f"Row {i+2} has no data (empty row_data)")
        data.append(row_data)
    return data

# Format content for RECENT_BLOG
def explore_all_content1(data, row_idx, category):
    try:
        row_data = data[row_idx - 2]
        if category in row_data:
            pieces = row_data[category]
            output = []
            for idx, content in enumerate(pieces, start=1):
                output.append(f"*Blog Post {idx}*:")
                output.append(content)
                output.append("")
                output.append(f"Content length: {len(content)} characters")
                output.append(f"Line count: {len(content.splitlines())}")
                output.append(f"Word Count: {len(content.split())}")
            return "\n".join(output)
        else:
            return f"Error: Category '{category}' not found in row {row_idx}."
    except IndexError:
        return f"Error: Row {row_idx} not found. Available rows: 2 to {len(data) + 1}"

# AI processing for RECENT_BLOG
def create_chain(category: str):
    if category not in PROMPT_TEMPLATES:
        raise ValueError(f"No prompt defined for category {category}")
    prompt = PromptTemplate(
        input_variables=["content"],
        template=PROMPT_TEMPLATES[category]
    )
    llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY)
    return prompt | llm | StrOutputParser()

async def process_row_with_langchain(content_output: str, category: str) -> str:
    chain = create_chain(category)
    if not content_output.strip() or "Error:" in content_output:
        return f"No valid content provided for {category}"
    try:
        response = await chain.ainvoke({"content": content_output})
        return response
    except Exception as e:
        print(f"LangChain error for category {category}: {e}")
        return f"Error processing row for {category}"

# Process a single category column
async def process_category(data, category, sheet_mgr, spreadsheet_id):
    column = COLUMN_TO_WRITE_URL_TO[category.upper()]
    values = []
    for row_idx in range(2, len(data) + 2):
        row_data = data[row_idx - 2]
        pieces = row_data.get(category, [])
        if not pieces:
            output = "no content"
        elif len(pieces) == 1:
            output = pieces[0]
        else:
            if category == "RECENT_BLOG":
                content_output = explore_all_content1(data, row_idx, category)
                if "Error:" in content_output:
                    output = "No valid content provided"
                else:
                    output = await process_row_with_langchain(content_output, category)
            else:
                # MODIFIED LOGIC: Take FIRST piece that meets threshold
                threshold = CATEGORY_THRESHOLDS.get(category.upper(), 0)
                first_qualifying_piece = None
                
                for piece in pieces:
                    if len(piece.split()) >= threshold:
                        first_qualifying_piece = piece
                        break  # Take the first one that meets threshold
                
                if first_qualifying_piece:
                    output = first_qualifying_piece
                else:
                    # Fallback: take the longest piece if none meet threshold
                    output = max(pieces, key=lambda p: len(p.split()))
                    
        values.append([output])
    
    range_name = f"{PROCESSED_SHEET_NAME}!{column}2:{column}{len(data) + 1}"
    try:
        sheet_mgr.service.spreadsheets().values().update(
            spreadsheetId=spreadsheet_id,
            range=range_name,
            valueInputOption='RAW',
            body={'values': values}
        ).execute()
        print(f"Updated column {column} for category {category} in {PROCESSED_SHEET_NAME}")
    except Exception as e:
        print(f"Error writing to {range_name}: {str(e)}")

# Main function
async def main():
    try:
        sheet_mgr = GoogleSheetsManager(CREDENTIALS_FILE)
        spreadsheet_id = sheet_mgr.extract_spreadsheet_id(SHEET_URL)
        print(f"Extracted spreadsheet ID: {spreadsheet_id}")
        data = await read_data_from_sheet(spreadsheet_id, sheet_mgr, CATEGORIES)
        for category in CATEGORIES:
            await process_category(data, category, sheet_mgr, spreadsheet_id)
        print("Processing complete.")
    except Exception as e:
        print(f"Error: {str(e)}")

await main()

Extracted spreadsheet ID: 1lmVQ8jnKWYowsfEG89FiiLIEAIkBs7uhshuOvMQ1tZc
Row 2 metadata: 
Row 2 has no data (empty row_data)
Row 3 metadata: PODCAST=0,ABOUT_US=3,SHOP=0,RECENT_BLOG=0,EBOOK=0,COURSES=0,TESTIMONIALS=1,WEBINAR=0,SERVICES=4
Debug - Row 3, ABOUT_US: Content preview: 'ABOUT US
Our
mission...
We stop burnout, disengagement, conflict, and violence in the workplace usin...'
Debug - Row 3, ABOUT_US: Found 3 pieces using delimiter '-----NEXT CONTENT FROM HERE-----'
Debug - Row 3, TESTIMONIALS: Content preview: 'All Posts
Search
Read a Participant Testimonial - September 2024
Zach Stone
Oct 1, 2024
1 min read
T...'
Debug - Row 3, TESTIMONIALS: No delimiter found, treating as single piece
Debug - Row 3, SERVICES: Content preview: 'For Executives
Use systems thinking and policy shifts to build resilient and performant work communi...'
Debug - Row 3, SERVICES: Found 4 pieces using delimiter '-----NEXT CONTENT FROM HERE-----'
Row 4 metadata: PODCAST=2,ABOUT_US=10,SHOP=3,RECENT_BLOG=3,EB